# Temperature and Hallucination — How Confidence Shapes Creativity

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Notebook Guide (connected to Lab 76: The Edge of Sleep)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/appendix-4.ipynb)

**What you will do:** change the sampling temperature and the sharpness of the model's underlying preferences to see when text generation stays predictable and when it turns into confident nonsense.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## Simulation

**Concept:** A parameter called "temperature" controls how much randomness enters a language model's choice of its next word, from safe and predictable at low settings to rare and sometimes incoherent at high ones. This is loosely analogous to the brain's own trade-off between precision (trusting strong predictions) and exploration (letting weaker associations surface)—though "temperature" is not the same thing as an AI "hallucination" (a confident but false statement), which can occur at any temperature.

**Relevant Labs:** Lab 76 (precision turned down: a brain state loosely comparable to high temperature), Lab 38 (hallucination in memory)

In [ ]:
# Temperature and Word Choice in Text Generation

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Simulated next-word probability distribution.
# Imagine the model is predicting the word after "The cat sat on the"
words = ['mat', 'floor', 'chair', 'roof', 'moon', 'theory', 'quantum',
         'banana', 'democracy', 'silence']
# Raw logits (pre-softmax scores) — higher = more likely
logits = np.array([5.0, 4.2, 3.5, 2.0, 0.5, -0.5, -1.0, -2.0, -3.0, -4.0])


def softmax_with_temperature(logits, temperature):
    """Apply temperature scaling and softmax."""
    scaled = logits / temperature
    exp_scaled = np.exp(scaled - np.max(scaled))  # numerical stability
    return exp_scaled / exp_scaled.sum()


temperatures = [0.1, 0.5, 1.0, 2.0, 5.0]
labels = ['T=0.1\n(Very focused)', 'T=0.5\n(Conservative)',
          'T=1.0\n(Balanced)', 'T=2.0\n(Creative)',
          'T=5.0\n(Near-random)']

fig, axes = plt.subplots(1, 5, figsize=(20, 5), sharey=True)

colours = ['#2c3e50', '#2980b9', '#27ae60', '#e67e22', '#e74c3c']

for i, (temp, label, colour) in enumerate(zip(temperatures, labels,
                                                colours)):
    probs = softmax_with_temperature(logits, temp)
    axes[i].barh(range(len(words)), probs, color=colour, alpha=0.8,
                 edgecolor='white', linewidth=0.5)
    axes[i].set_yticks(range(len(words)))
    axes[i].set_yticklabels(words, fontsize=10)
    axes[i].set_title(label, fontsize=11, fontweight='bold')
    axes[i].set_xlim(0, 1.05)

    # Annotate top probability
    max_idx = np.argmax(probs)
    top_prob = probs[max_idx]
    axes[i].annotate(f'{top_prob:.2%}', xy=(top_prob, max_idx),
                     xytext=(5, 0), textcoords='offset points',
                     fontsize=9, fontweight='bold')

axes[0].set_ylabel('Candidate next word', fontsize=12)
fig.suptitle('"The cat sat on the ___" — How Temperature Changes '
             'Prediction', fontsize=14, fontweight='bold')
fig.text(0.5, -0.02, 'Probability of being selected as next word →',
         ha='center', fontsize=12)

plt.tight_layout()
plt.savefig('temperature_hallucination.png', dpi=150,
            bbox_inches='tight')
plt.show()

# --- Demonstrate sampling ---
print("=" * 60)
print("SAMPLING DEMONSTRATION")
print("=" * 60)
print()
print('Prompt: "The cat sat on the ___"\n')

for temp, label in zip(temperatures, labels):
    probs = softmax_with_temperature(logits, temp)
    samples = np.random.choice(words, size=10, p=probs)
    print(f"{label.replace(chr(10), ' ')}")
    print(f"  10 samples: {', '.join(samples)}")
    print()

print("Key insight: temperature controls how much randomness the "
      "sampling step injects into word choice. Low temperature keeps "
      "the model near its most confident guess; high temperature lets "
      "it wander into rare, sometimes incoherent, word choices.")
print("This is not the same thing as an AI 'hallucination' in the "
      "usual sense: a hallucination is a confident but false "
      "statement, and those occur even at low temperature, because "
      "they come from what the model learned (or failed to learn), "
      "not only from how it samples.")
print("One way to picture the hypnagogic state (Lab 76) is as the "
      "brain's prediction engine running with its precision "
      "dialled down—loosely analogous to sampling at a higher "
      "temperature, though Lab 76 itself is described in terms of "
      "precision, not temperature.")

## Change the parameters

**temperature** works exactly as in the cell above. **sharpness** multiplies the original
logits before temperature is applied, simulating a model that is more (or less)
confident about its top choice to begin with. **n_samples** controls how many words are
drawn.
(If the sliders do not appear, run the cell again.)

In [ ]:
from ipywidgets import interact, FloatSlider, IntSlider

def explore(temperature=1.0, sharpness=1.0, n_samples=10):
    np.random.seed(42)
    scaled_logits = logits * sharpness
    probs = softmax_with_temperature(scaled_logits, temperature)

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.barh(range(len(words)), probs, color="steelblue", alpha=0.85)
    ax.set_yticks(range(len(words)))
    ax.set_yticklabels(words)
    ax.set_xlabel("Probability")
    ax.set_xlim(0, 1.05)
    ax.set_title(f"temperature = {temperature}, sharpness = {sharpness}")
    plt.tight_layout()
    plt.show()

    samples = np.random.choice(words, size=n_samples, p=probs)
    print(f"{n_samples} samples: {', '.join(samples)}")
    print(f"Probability of the single most likely word: {probs.max():.1%}")

interact(explore,
         temperature=FloatSlider(value=1.0, min=0.05, max=5.0, step=0.05),
         sharpness=FloatSlider(value=1.0, min=0.2, max=3.0, step=0.1),
         n_samples=IntSlider(value=10, min=5, max=30, step=5));

## Questions to think about

1. Set temperature to 0.1 and then to 5.0, keeping sharpness at 1.0. How does the printed probability of the single most likely word change, and how does that match the lab's description of low temperature as "safe, predictable" versus high temperature as "creative, surprising, sometimes nonsensical"?
2. Set sharpness to 3.0 (the model is much more confident in its top choice before temperature is applied) and try temperature=2.0. Compare the samples with sharpness=1.0 at the same temperature. Does a more confident underlying model resist high temperature better?
3. The lab draws a loose analogy between high temperature in AI and the hypnagogic state in Lab 76, where sensory prediction errors carry less weight ("precision" is turned down) because the thalamic gate relaxes. In this widget, what plays the role of that reduced weighting, which normally keeps the model's predictions in check?
4. Set temperature to its lowest value, 0.05. Do the samples ever include anything other than the single top-ranked word? What real-world downside might a language model have if it always ran at this temperature?

## Challenge

Lab 76 describes the Dali "spoon method" for catching creative material at the edge of
sleep, then writing it down immediately before it fades. Simulate an analogous "creative
harvesting" process: generate many high-temperature samples, then keep only the ones that
still look like plausible words, mimicking how a person captures the rare useful idea out of
a mostly chaotic state.

In [ ]:
# Example data: a small set of words judged "plausible" for this sentence. In a real
# system, this filter would be a language model or a human editor, not a fixed list --
# this is a simplified stand-in for that judgement.
plausible_words = {"mat", "floor", "chair", "roof"}  # example data

np.random.seed(7)
high_temp_samples = np.random.choice(words, size=200, p=softmax_with_temperature(logits, 4.0))
kept = [w for w in high_temp_samples if w in plausible_words]

print(f"Out of {len(high_temp_samples)} high-temperature samples, "
      f"{len(kept)} ({len(kept) / len(high_temp_samples):.0%}) passed the plausibility filter.")
print("This is a toy illustration of harvesting useful output from a noisy, high-temperature")
print("process -- the plausible_words set above is example data, not a trained filter.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "appendix-4.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")